In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

In [ ]:
# load data
data = torch.load('../data/bert_tokenized_tensors.pt')
print(data.keys())

C:\Users\reese\AppData\Local\Temp\ipykernel_30056\4015265430.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load('../data/bert_tokenized_tensors.pt')


dict_keys(['input_ids', 'attention_masks', 'labels'])


In [5]:
print(data['input_ids'].shape)
print(data['attention_masks'].shape)
print(data['labels'].shape)

torch.Size([562618, 256])
torch.Size([562618, 256])
torch.Size([562618])


In [ ]:
# Dataset

from torch.utils.data import Dataset

class PostsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # return super().__getitem__(index)
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item
    
    def __len__(self):
        return len(self.labels)

In [7]:
encodings = {
    "input_ids": data["input_ids"],
    "attention_masks": data["attention_masks"]
}

labels = data["labels"]

dataset = PostsDataset(encodings, labels)

In [40]:
# split dataset - 60% train, 20% validation, 20% test

from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# subset for testing
unused_subset, used_subset = train_test_split(
    range(len(dataset)), test_size=0.01, stratify=labels, random_state=42
)

train_idx, temp_idx = train_test_split(
    used_subset, test_size=0.4, stratify=[labels[i] for i in used_subset], random_state=42
)
## original
# train_idx, temp_idx = train_test_split(
#     range(len(dataset)), test_size=0.4, stratify=labels, random_state=42
# )

val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, stratify=[labels[i] for i in temp_idx], random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

In [41]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

3376
1125
1126


In [34]:
# DataLoader

from torch.utils.data import DataLoader

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
# load BERT model

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

c:\Users\reese\Documents\SUTD stuff\work stuff\term 6\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\reese\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2321.55it/s]
BertForSequenceClassifi

In [14]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# freeze the pretrained layers
for param in model.bert.parameters():
    param.requires_grad = False

# keep only the classification head trainable
for param in model.classifier.parameters():
    param.requires_grad = True

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Trainable parameters: 1538


In [48]:
# training function

from tqdm import tqdm

def train(model, dataloader, optimizer, device):
    model.train()

    total_loss = 0
    total_preds = []
    total_labels = []

    for step, batch in enumerate(tqdm(dataloader)):
        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_masks"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_masks,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        total_loss += loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        preds = torch.argmax(logits, dim=1)

        total_preds.extend(preds.detach().cpu().numpy())
        total_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(total_labels, total_preds)
    f1score = f1_score(total_labels, total_preds)

    return avg_loss, accuracy, f1score


In [49]:
# evaluation function

def evaluate(model, dataloader, device):
    model.eval()

    total_loss = 0
    total_preds = []
    total_labels = []

    with torch.no_grad():

        for batch in tqdm(dataloader):

            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_masks"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_masks,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            total_preds.extend(preds.detach().cpu().numpy())
            total_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(total_labels, total_preds)
    f1score = f1_score(total_labels, total_preds)

    return avg_loss, accuracy, f1score

In [50]:
# early stopping 

class EarlyStopping:
    def __init__(self, patience, delta=0, path="best_model.pt"):

        self.patience = patience # num of epochs without improvement before stopping
        self.delta = delta # min improvement threshold
        self.path = path

        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_score, model):

        if self.best_score is None:
            self.best_score = val_score
            self.save_checkpoint(model)

        elif val_score < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")

            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_score = val_score
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        print("Validation improved. Saving model...")
        torch.save(model.state_dict(), self.path)

In [ ]:
# ### this was a test, dont use ###
# training loop 

# early_stopping = EarlyStopping(patience=3, path="best_model.pt")

# num_epochs = 1
# train_losses = []
# val_losses = []
# train_accs = []
# val_accs = []

# for epoch in range(num_epochs):
#     print(f"\nEpoch {epoch+1}/{num_epochs}")

#     train_loss, train_acc, train_f1 = train(
#         model, train_loader, optimizer, device
#     )

#     val_loss, val_acc, val_f1 = evaluate(
#         model, val_loader, device
#     )

#     print(f"Train Loss: {train_loss:.4f}")
#     print(f"Train Acc:  {train_acc:.4f}")
#     print(f"Val Loss:   {val_loss:.4f}")
#     print(f"Val Acc:    {val_acc:.4f}")

#     train_losses.append(train_loss)
#     val_losses.append(val_loss)

#     train_accs.append(train_acc)
#     val_accs.append(val_acc)

#     # check if f1 score improved
#     early_stopping(val_f1, model)

#     if early_stopping.early_stop:
#         print("Early stopping triggered")
#         break



Epoch 1/1


100%|██████████| 352/352 [17:55<00:00,  3.06s/it]


Train Loss: 0.6283
Train Acc:  0.6745
Val Loss:   0.5840
Val Acc:    0.7057
Validation improved. Saving model...


In [ ]:
# print(train_f1, val_f1)

0.6654080389768575 0.742296918767507


In [51]:
# training loop

def training_loop(optimizer, early_stopping, num_epochs):
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    train_f1s = []
    val_f1s = []

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        train_loss, train_acc, train_f1 = train(
            model, train_loader, optimizer, device
        )

        val_loss, val_acc, val_f1 = evaluate(
            model, val_loader, device
        )

        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Acc:  {train_acc:.4f}")
        print(f"Train F1 score:    {train_f1:.4f}")
        print(f"Val Loss:   {val_loss:.4f}")
        print(f"Val Acc:    {val_acc:.4f}")
        print(f"Val F1 score:    {val_f1:.4f}")

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        train_accs.append(train_acc)
        val_accs.append(val_acc)

        train_f1s.append(train_f1)
        val_f1s.append(val_f1)

        # check if f1 score improved
        early_stopping(val_f1, model)

        if early_stopping.early_stop:
            print("Early stopping triggered")
            break
    
    return train_losses, val_losses, train_accs, val_accs, train_f1s, val_f1s


In [ ]:
# freeze the pretrained layers
for param in model.bert.parameters():
    param.requires_grad = False

# keep only the classification head trainable
for param in model.classifier.parameters():
    param.requires_grad = True

early_stopping = EarlyStopping(patience=3, path="best_model.pt")
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
train_losses_1, val_losses_1, train_accs_1, val_accs_1, train_f1s_1, val_f1s_1 = training_loop(
    optimizer, early_stopping, 1)

# unfreeze last 2 layers
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

early_stopping = EarlyStopping(patience=3, path="best_model.pt")
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
train_losses_2, val_losses_2, train_accs_2, val_accs_2, train_f1s_2, val_f1s_2 = training_loop(
    optimizer, early_stopping, 3)

# unfreeze last 4 layers
for param in model.bert.encoder.layer[-4:-2].parameters():
    param.requires_grad = True
    
early_stopping = EarlyStopping(patience=3, path="best_model.pt")
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)
train_losses_3, val_losses_3, train_accs_3, val_accs_3, train_f1s_3, val_f1s_3 = training_loop(
    optimizer, early_stopping, 3)


Epoch 1/1


100%|██████████| 352/352 [18:15<00:00,  3.11s/it]


Train Loss: 0.5320
Train Acc:  0.7749
Train F1 score:    0.7728
Val Loss:   0.5035
Val Acc:    0.7911
Val F1 score:    0.8047
Validation improved. Saving model...

Epoch 1/3


100%|██████████| 352/352 [18:06<00:00,  3.09s/it]


Train Loss: 0.1222
Train Acc:  0.9554
Train F1 score:    0.9541
Val Loss:   0.0687
Val Acc:    0.9774
Val F1 score:    0.9767
Validation improved. Saving model...

Epoch 2/3


100%|██████████| 352/352 [17:25<00:00,  2.97s/it]


Train Loss: 0.0648
Train Acc:  0.9784
Train F1 score:    0.9778
Val Loss:   0.0747
Val Acc:    0.9794
Val F1 score:    0.9790
Validation improved. Saving model...

Epoch 3/3


 44%|████▍     | 469/1055 [39:57<50:55,  5.21s/it]  

In [ ]:
# plot graphs

iters = list(range(1, 8))

total_train_losses = train_losses_1 + train_losses_2 + train_losses_3
total_train_accs = train_accs_1 + train_accs_2 + train_accs_3
total_train_f1s = train_f1s_1 + train_f1s_2 + train_f1s_3
total_val_losses = val_losses_1 + val_losses_2 + val_losses_3
total_val_accs = val_accs_1 + val_accs_2 + val_accs_3
total_val_f1s = val_f1s_1 + val_f1s_2 + val_f1s_3

fig, ax = plt.subplots(1, 3)
ax[0,0].plot(iters, total_train_losses, marker='o', label='Training Loss')
ax[0,0].plot(iters, total_val_losses, marker='s', label='Validation Loss')
ax[0,1].plot(iters, total_train_accs, marker='o', label='Training Accuracy')
ax[0,1].plot(iters, total_val_accs, marker='s', label='Validation Accuracy')
ax[0,2].plot(iters, total_train_f1s, marker='o', label='Training F1 Score')
ax[0,2].plot(iters, total_val_f1s, marker='s', label='Validation F1 Score')

plt.legend()
plt.show()


<class 'list'>


In [ ]:
def evaluate_on_testing_set(y_test, y_pred):
  # Calculate AUC
  print("AUC is: ", roc_auc_score(y_test, y_pred))

  # print out recall and precision
  print(classification_report(y_test, y_pred))

  # print out confusion matrix
  print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred))

  # # calculate points for ROC curve
  fpr, tpr, thresholds = roc_curve(y_test, y_pred)

  # Plot ROC curve
  plt.plot(fpr, tpr, label='ROC curve (area = %0.3f)' % roc_auc_score(y_test, y_pred))
  plt.plot([0, 1], [0, 1], 'k--')  # random predictions curve
  plt.xlim([0.0, 1.0])
  plt.ylim([0.0, 1.0])
  plt.xlabel('False Positive Rate or (1 - Specifity)')
  plt.ylabel('True Positive Rate or (Sensitivity)')
  plt.title('Receiver Operating Characteristic')

In [ ]:
# testing

model.eval()

test_loss = 0
test_preds = []
test_labels = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_masks"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_masks,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        test_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        test_preds.extend(preds.detach().cpu().numpy())
        test_labels.extend(labels.detach().cpu().numpy())

In [ ]:
# testing metrics

print("Test Loss:", test_loss)
evaluate_on_testing_set(test_labels, test_preds)